In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from itertools import combinations

print("Imports successful")

Imports successful


In [2]:
# Cell 2 — Haversine & Vincenty Great Circle Formulas
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate great circle distance between two points
    Returns distance in kilometers
    """
    R = 6371.0  # Earth radius in km
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

def initial_bearing(lat1, lon1, lat2, lon2):
    """Calculate initial bearing from point 1 to point 2 in degrees"""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return np.degrees(np.arctan2(x, y)) % 360

# Major airport coordinates
airports = {
    'JFK':  {'lat': 40.6413, 'lon': -73.7781, 'city': 'New York'},
    'LAX':  {'lat': 33.9425, 'lon': -118.4081,'city': 'Los Angeles'},
    'LHR':  {'lat': 51.4700, 'lon': -0.4543,  'city': 'London'},
    'NRT':  {'lat': 35.7647, 'lon': 140.3864, 'city': 'Tokyo'},
    'DXB':  {'lat': 25.2532, 'lon': 55.3657,  'city': 'Dubai'},
    'SYD':  {'lat': -33.9399,'lon': 151.1753, 'city': 'Sydney'},
    'ORD':  {'lat': 41.9742, 'lon': -87.9073, 'city': 'Chicago'},
    'CDG':  {'lat': 49.0097, 'lon': 2.5479,   'city': 'Paris'},
    'SEA':  {'lat': 47.4502, 'lon': -122.3088,'city': 'Seattle'},
    'BOS':  {'lat': 42.3656, 'lon': -71.0096, 'city': 'Boston'}
}

# Compute distances between all airport pairs
print(f"{'Route':<12} {'Distance (km)':>15} {'Bearing (°)':>12}")
print("-" * 42)
for (a1, a2) in [('BOS','LHR'),('JFK','NRT'),('LAX','SYD'),('SEA','DXB')]:
    d = haversine(airports[a1]['lat'], airports[a1]['lon'],
                  airports[a2]['lat'], airports[a2]['lon'])
    b = initial_bearing(airports[a1]['lat'], airports[a1]['lon'],
                        airports[a2]['lat'], airports[a2]['lon'])
    print(f"{a1}-{a2:<8} {d:>15,.1f} {b:>12.1f}")

Route          Distance (km)  Bearing (°)
------------------------------------------
BOS-LHR              5,240.4         53.3
JFK-NRT             10,830.3        332.6
LAX-SYD             12,060.9        241.0
SEA-DXB             11,927.5          2.2


In [3]:
# Cell 3 — Generate Great Circle Arc Points for Plotting
def great_circle_points(lat1, lon1, lat2, lon2, n=100):
    """Generate n points along great circle path for visualization"""
    points = []
    for t in np.linspace(0, 1, n):
        lat1r, lon1r, lat2r, lon2r = map(np.radians, [lat1, lon1, lat2, lon2])
        d = 2 * np.arcsin(np.sqrt(
            np.sin((lat2r-lat1r)/2)**2 +
            np.cos(lat1r)*np.cos(lat2r)*np.sin((lon2r-lon1r)/2)**2
        ))
        if d == 0:
            return [(lat1, lon1)]
        A = np.sin((1-t)*d) / np.sin(d)
        B = np.sin(t*d)     / np.sin(d)
        x = A*np.cos(lat1r)*np.cos(lon1r) + B*np.cos(lat2r)*np.cos(lon2r)
        y = A*np.cos(lat1r)*np.sin(lon1r) + B*np.cos(lat2r)*np.sin(lon2r)
        z = A*np.sin(lat1r)               + B*np.sin(lat2r)
        lat = np.degrees(np.arctan2(z, np.sqrt(x**2+y**2)))
        lon = np.degrees(np.arctan2(y, x))
        points.append((lat, lon))
    return points

# Plot great circle routes on world map
routes = [('BOS','LHR'), ('JFK','NRT'), ('LAX','SYD'), ('SEA','DXB'), ('ORD','CDG')]
colors = ['cyan', 'orange', 'lime', 'crimson', 'violet']

fig = go.Figure()

# Airport markers
for code, info in airports.items():
    fig.add_trace(go.Scattergeo(
        lat=[info['lat']], lon=[info['lon']],
        mode='markers+text',
        marker=dict(size=8, color='white'),
        text=[code], textposition='top right',
        textfont=dict(size=11, color='white'),
        showlegend=False
    ))

# Route arcs
for (a1, a2), color in zip(routes, colors):
    pts = great_circle_points(
        airports[a1]['lat'], airports[a1]['lon'],
        airports[a2]['lat'], airports[a2]['lon']
    )
    lats, lons = zip(*pts)
    d = haversine(airports[a1]['lat'], airports[a1]['lon'],
                  airports[a2]['lat'], airports[a2]['lon'])
    fig.add_trace(go.Scattergeo(
        lat=lats, lon=lons,
        mode='lines',
        line=dict(width=2, color=color),
        name=f'{a1}→{a2} ({d:,.0f} km)'
    ))

fig.update_layout(
    title='Great Circle Routes — Major International Airports',
    geo=dict(
        showland=True, landcolor='rgb(30,30,30)',
        showocean=True, oceancolor='rgb(10,10,50)',
        showcoastlines=True, coastlinecolor='gray',
        projection_type='natural earth',
        bgcolor='rgb(10,10,30)'
    ),
    template='plotly_dark',
    width=950, height=550
)
fig.show()

In [4]:
# Cell 4 — Wind-Corrected Fuel Burn Model
def fuel_burn(distance_km, wind_component_kts, aircraft='B737'):
    """
    Estimate fuel burn with wind correction
    wind_component_kts: positive = tailwind, negative = headwind
    """
    specs = {
        'B737': {'cruise_speed_kts': 450, 'burn_rate_kgh': 2500, 'capacity_kg': 20000},
        'B777': {'cruise_speed_kts': 490, 'burn_rate_kgh': 6800, 'capacity_kg': 145000},
        'A320': {'cruise_speed_kts': 430, 'burn_rate_kgh': 2400, 'capacity_kg': 18000}
    }
    s = specs[aircraft]
    
    # Ground speed with wind
    ground_speed = s['cruise_speed_kts'] + wind_component_kts
    
    # Distance in nautical miles
    distance_nm = distance_km / 1.852
    
    # Flight time in hours
    flight_time_h = distance_nm / ground_speed
    
    # Fuel burn
    fuel_kg = flight_time_h * s['burn_rate_kgh']
    fuel_feasible = fuel_kg <= s['capacity_kg']
    
    return {
        'flight_time_h': round(flight_time_h, 2),
        'fuel_kg':        round(fuel_kg, 1),
        'ground_speed':   round(ground_speed, 1),
        'feasible':       fuel_feasible
    }

# Compare routes under different wind conditions
print(f"{'Route':<10} {'Wind':>8} {'Aircraft':>8} {'Time (h)':>10} {'Fuel (kg)':>12} {'OK':>6}")
print("-" * 58)

test_cases = [
    ('BOS', 'LHR', -30, 'B737'),   # headwind transatlantic
    ('BOS', 'LHR',  50, 'B737'),   # tailwind transatlantic
    ('JFK', 'NRT', -20, 'B777'),   # transpacific headwind
    ('JFK', 'NRT',  40, 'B777'),   # transpacific tailwind
    ('SEA', 'DXB',   0, 'B777'),   # no wind
]

for a1, a2, wind, aircraft in test_cases:
    d = haversine(airports[a1]['lat'], airports[a1]['lon'],
                  airports[a2]['lat'], airports[a2]['lon'])
    result = fuel_burn(d, wind, aircraft)
    wind_str = f"{wind:+d} kts"
    print(f"{a1}-{a2:<5} {wind_str:>8} {aircraft:>8} "
          f"{result['flight_time_h']:>10.2f} "
          f"{result['fuel_kg']:>12,.0f} "
          f"{'YES' if result['feasible'] else 'NO':>6}")

Route          Wind Aircraft   Time (h)    Fuel (kg)     OK
----------------------------------------------------------
BOS-LHR    -30 kts     B737       6.74       16,843    YES
BOS-LHR    +50 kts     B737       5.66       14,148    YES
JFK-NRT    -20 kts     B777      12.44       84,608    YES
JFK-NRT    +40 kts     B777      11.03       75,030    YES
SEA-DXB     +0 kts     B777      13.14       89,376    YES


In [5]:
# Cell 5 — ETOPS Constraint Modeling
def etops_radius_km(etops_minutes, cruise_speed_kts=450):
    """
    Calculate ETOPS diversion radius in km
    etops_minutes: 60, 120, 180, or 240
    """
    radius_nm = (etops_minutes / 60) * cruise_speed_kts
    return radius_nm * 1.852

def check_etops(lat1, lon1, lat2, lon2, alternate_airports, etops_min=180):
    """Check if route is ETOPS compliant given alternate airports"""
    radius_km = etops_radius_km(etops_minutes=etops_min)
    pts = great_circle_points(lat1, lon1, lat2, lon2, n=50)
    
    violations = []
    for i, (plat, plon) in enumerate(pts):
        covered = False
        for code, info in alternate_airports.items():
            d = haversine(plat, plon, info['lat'], info['lon'])
            if d <= radius_km:
                covered = True
                break
        if not covered:
            violations.append(i)
    
    compliance = len(violations) == 0
    return {
        'etops_minutes':   etops_min,
        'radius_km':       round(radius_km, 1),
        'total_points':    len(pts),
        'violations':      len(violations),
        'compliant':       compliance
    }

# Alternate airports for ETOPS
alternates = {
    'KEF': {'lat': 63.9850, 'lon': -22.6056, 'city': 'Reykjavik'},
    'YYR': {'lat': 53.3192, 'lon': -60.4256, 'city': 'Goose Bay'},
    'SNN': {'lat': 52.7019, 'lon': -8.9248,  'city': 'Shannon'},
    'AZR': {'lat': 27.8376, 'lon': -0.1864,  'city': 'Adrar'}
}

print("=== ETOPS Compliance Check: BOS → LHR ===\n")
for etops in [60, 120, 180]:
    result = check_etops(
        airports['BOS']['lat'], airports['BOS']['lon'],
        airports['LHR']['lat'], airports['LHR']['lon'],
        alternates, etops_min=etops
    )
    status = 'COMPLIANT' if result['compliant'] else 'NON-COMPLIANT'
    print(f"  ETOPS-{etops:>3}: radius={result['radius_km']:>7,.1f} km  "
          f"violations={result['violations']:>3}  → {status}")

=== ETOPS Compliance Check: BOS → LHR ===

  ETOPS- 60: radius=  833.4 km  violations= 25  → NON-COMPLIANT
  ETOPS-120: radius=1,666.8 km  violations=  0  → COMPLIANT
  ETOPS-180: radius=2,500.2 km  violations=  0  → COMPLIANT


In [6]:
# Cell 6 — Optimal Route Comparison & Export
import os

results = []
for (a1, a2) in [('BOS','LHR'),('JFK','NRT'),('LAX','SYD'),('SEA','DXB'),('ORD','CDG')]:
    d = haversine(airports[a1]['lat'], airports[a1]['lon'],
                  airports[a2]['lat'], airports[a2]['lon'])
    b = initial_bearing(airports[a1]['lat'], airports[a1]['lon'],
                        airports[a2]['lat'], airports[a2]['lon'])
    fb_hw = fuel_burn(d, -30, 'B777')  # headwind
    fb_tw = fuel_burn(d,  40, 'B777')  # tailwind
    
    results.append({
        'route':              f'{a1}-{a2}',
        'distance_km':        round(d, 1),
        'initial_bearing_deg':round(b, 1),
        'time_headwind_h':    fb_hw['flight_time_h'],
        'time_tailwind_h':    fb_tw['flight_time_h'],
        'fuel_headwind_kg':   fb_hw['fuel_kg'],
        'fuel_tailwind_kg':   fb_tw['fuel_kg'],
        'fuel_saving_kg':     round(fb_hw['fuel_kg'] - fb_tw['fuel_kg'], 1)
    })

df_routes = pd.DataFrame(results)
print(df_routes[['route','distance_km','time_tailwind_h',
                  'fuel_tailwind_kg','fuel_saving_kg']].to_string(index=False))

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-06-great-circle-routing\outputs'
os.makedirs(output_dir, exist_ok=True)
df_routes.to_csv(f'{output_dir}\\route_analysis.csv', index=False)
print(f"\nExported to outputs/")

  route  distance_km  time_tailwind_h  fuel_tailwind_kg  fuel_saving_kg
BOS-LHR       5240.4             5.34           36304.0          5524.5
JFK-NRT      10830.3            11.03           75029.8         11417.5
LAX-SYD      12060.9            12.29           83555.0         12714.9
SEA-DXB      11927.5            12.15           82630.5         12574.3
ORD-CDG       6664.7             6.79           46171.6          7026.1

Exported to outputs/
